[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q2/00b_matrix_quality_check.ipynb)

# R1-Q2 Week 1 (continued) — Matrix quality check

This notebook inspects the filtered shoot and root expression matrices
that `00_question_orientation.ipynb` produced. It does not change them,
re-filter them, or write anything to disk — it reads what's there and
reports back.

The point: a matrix can pass each per-step check in N0 (the log
transform worked, the MAD distribution looks right, the filtered shapes
are plausible) and still not be in shape for downstream co-expression
work. The diagnostics here are the next layer of "is this fit for
purpose" — sample-level structure, gene-level constancy, control
probes, and the whole-matrix correlation picture.

By the end of this notebook you will have:

- A per-condition sample count showing whether any tissue × stress cell
  is thinly populated.
- A sample-level PCA per tissue showing whether any samples sit far
  from the rest of their condition (outlier candidates).
- A pairwise sample-correlation heatmap per tissue showing whether
  replicates and conditions cluster as expected.
- A flag for genes that survived the MAD filter but have effectively
  zero range across samples.
- A count of control probes (`AFFX-*`, including the bacterial spike-in
  controls) that survived the filter, with a named decision for you to
  make.
- A four-number summary of the pairwise gene-gene correlation
  distribution per tissue, and a synthesis cell that names any concerns
  and points to next steps.

If the diagnostics come back clean, you proceed to Week 2's
`01_wgcna.ipynb`. If they surface a problem, you go back to N0, revise
the relevant pre-commit and re-run, then re-run this notebook to
confirm the fix. The synthesis at the end of this notebook is where
that branch decision gets made.

In [ ]:
# There are three patterns for installing irilab2026 from GitHub, depending on your needs. 

# The first is for a fresh, complete runtime. This installs irilab2026 and every dependency it declares. 
# It skips anything pip already sees as installed. This is ideal for a new environment or when you want 
# to be sure everything is up to date.

#!pip install git+https://github.com/geraldmc/irilab2026.git@main
 
# The second is for code iteration only. It reinstalls irilab2026 itself, ignoring its dep list entirely. 
# The dep tree is left exactly as it is. This is ideal for iterating on irilab2026 itself, when you know 
# the deps are already satisfied and don't want to waste time reinstalling them.
# 
!pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

# The third is for developers who want to work with the code directly and see changes reflected 
# immediately without reinstalling.

#!pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main
#!pip install git+https://github.com/geraldmc/irilab2026.git@main

In [ ]:
# Mount Drive, set up irilab2026, point to the R1-Q2 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r1_q2")
print(f"Output directory: {OUTPUT_DIR}")

## 2) Load and orient

Three pieces of input feed every diagnostic that follows:

1. **The two filtered parquets** (`shoot_filtered.parquet`,
   `root_filtered.parquet`) — what N0 wrote at the end of its Section
   3.
2. **The sample metadata** — which GSM belongs to which tissue, which
   stress condition, and which replicate. Comes from
   `iri.atgenexpress_metadata()`, the same function N0 used.
3. **`precommit.json`** — your Week 1 pre-commits. We don't act on
   these here, but we'll echo the `gene_filtering` block so the rest
   of the notebook has the filter choices visible on screen.

Load each in turn, then confirm the sample columns in the two
parquets line up with the metadata's GSM index. If any of these
loads fail, the error message tells you what to do.

In [ ]:
import pandas as pd

shoot_path = OUTPUT_DIR / "shoot_filtered.parquet"
root_path = OUTPUT_DIR / "root_filtered.parquet"

try:
    shoot = pd.read_parquet(shoot_path)
    root = pd.read_parquet(root_path)
except FileNotFoundError as exc:
    raise FileNotFoundError(
        f"\nCould not find a filtered matrix.\n"
        f"  Looked for: {shoot_path}\n"
        f"              {root_path}\n\n"
        f"Run `00_question_orientation.ipynb` to completion first — "
        f"its Section 3 writes both parquets to this location.\n"
    ) from None

print(f"Shoot matrix: {shoot.shape[0]:>5} genes × {shoot.shape[1]:>3} samples")
print(f"Root matrix:  {root.shape[0]:>5} genes × {root.shape[1]:>3} samples")

In [ ]:
# Same metadata function N0 used. The SOFT cache should be warm from
# N0's run; if it isn't, this will pull from GEO on first call.
metadata = iri.atgenexpress_metadata()

print(f"Metadata: {metadata.shape[0]} samples × {metadata.shape[1]} columns")
print(f"Columns:  {list(metadata.columns)}")
print(f"\nStress counts:")
print(metadata["stress"].value_counts().sort_index())
print(f"\nTissue counts:")
print(metadata["tissue"].value_counts())

In [ ]:
# Every sample column in the parquets should be a GSM the metadata
# knows about, and the tissue labels should match expectations.
shoot_gsms = set(shoot.columns)
root_gsms = set(root.columns)
metadata_gsms = set(metadata.index)

shoot_missing = shoot_gsms - metadata_gsms
root_missing = root_gsms - metadata_gsms

print(f"Shoot samples in metadata: {len(shoot_gsms - shoot_missing):>3} of {len(shoot_gsms)}")
print(f"Root samples in metadata:  {len(root_gsms - root_missing):>3} of {len(root_gsms)}")
if shoot_missing:
    print(f"  Shoot GSMs not in metadata: {sorted(shoot_missing)}")
if root_missing:
    print(f"  Root GSMs not in metadata:  {sorted(root_missing)}")

# Tissue label match: every shoot column should be metadata-tagged "shoot",
# every root column should be metadata-tagged "root".
shoot_tissues = metadata.loc[list(shoot_gsms & metadata_gsms), "tissue"].value_counts()
root_tissues = metadata.loc[list(root_gsms & metadata_gsms), "tissue"].value_counts()
print(f"\nShoot-matrix samples by tissue: {dict(shoot_tissues)}")
print(f"Root-matrix samples by tissue:  {dict(root_tissues)}")

assert not shoot_missing and not root_missing, "Some samples are not in metadata — investigate before continuing."
assert set(shoot_tissues.index) == {"shoot"}, "Shoot matrix contains non-shoot samples."
assert set(root_tissues.index) == {"root"}, "Root matrix contains non-root samples."
print("\nAlignment OK.")

### 2.4) Echo the pre-commit you're checking against

Read `precommit.json` and print the `gene_filtering` block. The
diagnostics that follow are checking whether the filter recorded here
actually produced matrices that are fit for WGCNA — so it helps to
have the filter choices on screen as you read the rest of the
notebook.

In [ ]:
import json

precommit_path = OUTPUT_DIR / "precommit.json"
try:
    with open(precommit_path) as f:
        precommit = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(
        f"\nCould not find your pre-commit file.\n"
        f"  Expected at: {precommit_path}\n\n"
        f"Run `00_question_orientation.ipynb` to completion first — "
        f"its final section writes `precommit.json` to this location.\n"
    ) from None

gene_filtering = precommit.get("gene_filtering", {})
print("gene_filtering pre-commit:")
print(json.dumps(gene_filtering, indent=2))

You now have the two filtered matrices, the sample metadata, and your
gene-filtering pre-commit in scope. The next section looks at
sample-level structure: distribution per condition, PCA, and pairwise
correlation. That's the layer most likely to surface what was wrong
with the matrices Notebook 1 tried (and failed) to build a co-expression
network on.

## 3) Sample-level structure

Three diagnostics in this section, each looking at a different facet
of how samples relate to each other within a tissue:

- **Per-condition counts.** Is any tissue × stress cell thinly populated?
  A condition with very few samples can pass through the MAD filter
  without contributing meaningfully to gene-coexpression patterns —
  worth knowing before reading downstream module results.
- **Sample-level PCA.** Do the samples within a tissue cluster the way
  metadata suggests they should? Outlier samples — points sitting far
  from the rest of their condition — can degrade scale-free fit across
  an entire matrix without changing the median correlation distribution
  much. PCA is the easiest place to spot them.
- **Pairwise sample correlation heatmap.** Do replicates cluster
  together? Do related conditions show block structure? The
  sample × sample correlation matrix is a denser view of the same
  question the PCA scatter asks more loosely.

Run all three per tissue. Anything notable goes into the synthesis at
the end of the notebook.

### 3.1) Sample distribution per condition

A counts table showing how many samples each (tissue × stress) cell
contains. The total per tissue is fixed (124 each), so this surfaces
asymmetries — whether some stresses are over- or under-represented
relative to others, and whether shoot and root have the same coverage
per stress.

In [ ]:
import numpy as np

shoot_meta = metadata.loc[shoot.columns]
root_meta = metadata.loc[root.columns]

counts = pd.DataFrame({
    "shoot": shoot_meta["stress"].value_counts(),
    "root": root_meta["stress"].value_counts(),
}).sort_index().fillna(0).astype(int)
counts["both"] = counts["shoot"] + counts["root"]

print("Samples per (stress × tissue):")
print(counts)

cell_values = counts[["shoot", "root"]].values.flatten()
print(f"\nThinnest tissue × stress cell: {cell_values.min()} samples")
print(f"Thickest tissue × stress cell: {cell_values.max()} samples")
print(f"Median cell:                   {int(np.median(cell_values))} samples")

# Stresses where shoot and root coverage differ — usually they shouldn't.
asymmetric = counts[counts["shoot"] != counts["root"]]
if len(asymmetric):
    print(f"\nStresses with shoot ≠ root sample count:")
    print(asymmetric[["shoot", "root"]])
else:
    print("\nShoot and root sample counts match for every stress.")

### 3.2) Sample-level PCA per tissue

Run PCA on each tissue's filtered matrix (samples as observations,
genes as features). Plot the first two principal components as a
scatter, colored by stress condition. Print the proportion of variance
each of the top five PCs captures.

What to look at:

- **Cluster structure.** Do points of the same stress sit near each
  other? Replicates from the same stress at the same time point
  should cluster tightly. If they don't, sample-level noise is high.
- **Position of any single point.** Is any point conspicuously far
  from the rest of its color group? That's an outlier candidate —
  worth investigating before trusting downstream network-level
  patterns.
- **Variance captured by PC1.** If PC1 captures most of the variance
  (say > 60%), one axis is dominating the matrix. Could be a strong
  biological signal (e.g., one stress producing a much bigger response
  than the others) or could be a technical artifact (e.g., a batch
  effect aligned with one stress).

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

def run_pca(matrix, n_components=5):
    """PCA on samples (transpose so rows are samples, columns are genes)."""
    pca = PCA(n_components=n_components)
    coords = pca.fit_transform(matrix.T.values)
    return pca, coords

shoot_pca, shoot_coords = run_pca(shoot)
root_pca, root_coords = run_pca(root)

# Set up a shared color map across both panels so the legend matches.
stresses = sorted(set(shoot_meta["stress"]) | set(root_meta["stress"]))
color_map = dict(zip(stresses, plt.cm.tab10(np.linspace(0, 1, len(stresses)))))

fig, (ax_shoot, ax_root) = plt.subplots(1, 2, figsize=(12, 5))

for stress in stresses:
    mask = (shoot_meta["stress"] == stress).values
    ax_shoot.scatter(shoot_coords[mask, 0], shoot_coords[mask, 1],
                     color=color_map[stress], label=stress, alpha=0.75, s=35)
    mask = (root_meta["stress"] == stress).values
    ax_root.scatter(root_coords[mask, 0], root_coords[mask, 1],
                    color=color_map[stress], label=stress, alpha=0.75, s=35)

ax_shoot.set_title(f"Shoot PCA (n={shoot.shape[1]} samples, {shoot.shape[0]} genes)")
ax_shoot.set_xlabel(f"PC1 ({shoot_pca.explained_variance_ratio_[0]:.1%} of variance)")
ax_shoot.set_ylabel(f"PC2 ({shoot_pca.explained_variance_ratio_[1]:.1%} of variance)")
ax_shoot.axhline(0, color="lightgrey", linewidth=0.5, zorder=0)
ax_shoot.axvline(0, color="lightgrey", linewidth=0.5, zorder=0)

ax_root.set_title(f"Root PCA (n={root.shape[1]} samples, {root.shape[0]} genes)")
ax_root.set_xlabel(f"PC1 ({root_pca.explained_variance_ratio_[0]:.1%} of variance)")
ax_root.set_ylabel(f"PC2 ({root_pca.explained_variance_ratio_[1]:.1%} of variance)")
ax_root.axhline(0, color="lightgrey", linewidth=0.5, zorder=0)
ax_root.axvline(0, color="lightgrey", linewidth=0.5, zorder=0)

handles, labels = ax_shoot.get_legend_handles_labels()
fig.legend(handles, labels, loc="center right", bbox_to_anchor=(1.08, 0.5),
           title="Stress", frameon=False)

plt.tight_layout()
plt.show()

print("Variance explained by PC (first five):")
print(f"  {'PC':>4}  {'shoot':>8}   {'root':>8}")
for i in range(5):
    print(f"  PC{i+1:<2}  {shoot_pca.explained_variance_ratio_[i]:>7.1%}   "
          f"{root_pca.explained_variance_ratio_[i]:>7.1%}")
print(f"  {'sum':>4}  {shoot_pca.explained_variance_ratio_[:5].sum():>7.1%}   "
      f"{root_pca.explained_variance_ratio_[:5].sum():>7.1%}")

### 3.3) Pairwise sample correlation heatmap per tissue

Compute the sample × sample Pearson correlation matrix within each
tissue, with samples ordered by stress, then by time point, then by
replicate. Plot as a heatmap, one per tissue.

What healthy structure looks like:

- **Bright diagonal** (correlation 1.0; each sample with itself).
- **Bright blocks along the diagonal** within each (stress, time) group
  — replicates from the same condition correlated near 1.0 with each
  other.
- **Slightly dimmer off-diagonal blocks** between different stresses —
  still high (these are all within-tissue samples), but not as high as
  within-condition.

Things that would be concerning:

- **A row/column noticeably darker than its neighbors.** That sample
  correlates poorly with everything, including its replicates. Outlier
  candidate — same role PCA plays in 3.2 from a different angle.
- **No visible block structure.** Suggests replicates aren't clustering
  with each other, which means either the metadata is wrong or the
  replicate variation is unusually high.

In [ ]:
def plot_sample_correlation(matrix, meta, title, ax):
    """Per-tissue sample correlation heatmap, ordered by (stress, time, replicate)."""
    ordered = meta.sort_values(["stress", "time_h", "replicate"])
    matrix_ordered = matrix[ordered.index]
    corr = matrix_ordered.corr()

    im = ax.imshow(corr.values, cmap="viridis", vmin=0.7, vmax=1.0, aspect="auto")
    ax.set_title(f"{title}  (vmin=0.7, vmax=1.0)")

    # Stress block boundaries and centered labels.
    stress_at_pos = ordered["stress"].values
    boundaries = []
    label_positions = []
    label_names = []
    start = 0
    for i in range(1, len(stress_at_pos) + 1):
        if i == len(stress_at_pos) or stress_at_pos[i] != stress_at_pos[i - 1]:
            label_positions.append((start + i - 1) / 2)
            label_names.append(stress_at_pos[i - 1])
            if i < len(stress_at_pos):
                boundaries.append(i)
            start = i

    for b in boundaries:
        ax.axhline(b - 0.5, color="white", linewidth=0.5)
        ax.axvline(b - 0.5, color="white", linewidth=0.5)

    ax.set_xticks(label_positions)
    ax.set_xticklabels(label_names, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(label_positions)
    ax.set_yticklabels(label_names, fontsize=8)
    return im

fig, (ax_shoot, ax_root) = plt.subplots(1, 2, figsize=(14, 6))

im_shoot = plot_sample_correlation(shoot, shoot_meta, "Shoot", ax_shoot)
im_root = plot_sample_correlation(root, root_meta, "Root", ax_root)

fig.colorbar(im_root, ax=[ax_shoot, ax_root], label="Pearson r", shrink=0.8)
plt.show()

# Per-tissue floor: the lowest off-diagonal correlation in the matrix.
def correlation_floor(matrix):
    corr = matrix.corr().values
    np.fill_diagonal(corr, np.nan)
    return np.nanmin(corr)

print(f"Lowest sample-sample correlation:")
print(f"  shoot:  {correlation_floor(shoot):.3f}")
print(f"  root:   {correlation_floor(root):.3f}")